# 05 — GraphRAG: Knowledge Graph + RAG

GraphRAG extrai **entidades e relacoes** dos documentos para criar um knowledge graph.
A busca considera tanto similaridade textual quanto relacoes entre conceitos.

## Quando usar GraphRAG?

- Documentos com muitas entidades inter-relacionadas
- Queries sobre relacoes: "Qual a diferenca entre X e Y?"
- Sintese de informacoes de multiplos documentos
- Raciocinio multi-hop: "Se A implica B e B implica C..."

## Diagrama

```
Documentos
    |
[Extracao de Entidades + Relacoes] (via LLM)
    |
Knowledge Graph (NetworkX)
    |                |
[Graph Search]  [Vector Search]
    |                |
    [Score Fusion]
         |
    Subgrafo + Chunks Relevantes
         |
       [LLM]
         |
      Resposta
```

In [ ]:
import sys
sys.path.insert(0, '..')

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import ollama
import json
import re
import httpx
from pathlib import Path
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

try:
    r = httpx.get('http://localhost:11434/api/tags')
    modelos = [m['name'] for m in r.json().get('models', [])]
    LLM = 'llama3.2' if any('llama3.2' in m for m in modelos) else (modelos[0] if modelos else None)
    print(f'LLM: {LLM}')
except:
    LLM = None

print('GraphRAG pronto!')

## 5.1 Extracao de Entidades e Relacoes

In [ ]:
EXTRACT_PROMPT = """Extraia entidades e relacoes do texto abaixo.
Retorne APENAS JSON valido no formato:
{"entities": ["entidade1", "entidade2"], "relations": [["entidade1", "relacao", "entidade2"]]}

Texto:
{texto}

JSON:"""

def extrair_grafo(texto: str) -> dict:
    """Extrai entidades e relacoes de um texto usando o LLM."""
    if not LLM:
        return {'entities': [], 'relations': []}
    
    prompt = EXTRACT_PROMPT.format(texto=texto[:1000])  # limitar tamanho
    response = ollama.chat(model=LLM, messages=[{'role': 'user', 'content': prompt}])
    
    # Extrair JSON da resposta
    text = response['message']['content']
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass
    return {'entities': [], 'relations': []}

# Textos de exemplo
textos = [
    """HNSW e um algoritmo de indexacao que usa um grafo hierarquico.
    O parametro m controla o numero de conexoes. HNSW e usado pelo Qdrant.
    Qdrant e um banco de dados vetorial que suporta busca semantica.""",
    
    """Embeddings sao criados por modelos como BERT e sentence-transformers.
    Esses modelos usam a arquitetura Transformer. 
    Os embeddings captura relacoes semanticas entre textos.
    Sentence-transformers e baseado no modelo BERT pre-treinado.""",
    
    """RAG usa embeddings para buscar documentos relevantes.
    Os documentos sao armazenados no Qdrant.
    O LLM recebe os documentos e gera uma resposta fundamentada.
    LangChain e LlamaIndex sao frameworks para construir pipelines RAG.""",
]

print('Extraindo entidades e relacoes dos textos...')
grafos_extraidos = [extrair_grafo(t) for t in textos]

for i, g in enumerate(grafos_extraidos):
    print(f'\nTexto {i+1}:')
    print(f'  Entidades: {g["entities"][:5]}')
    print(f'  Relacoes: {g["relations"][:3]}')

## 5.2 Construir o Knowledge Graph

In [ ]:
# Construir grafo com NetworkX
G = nx.DiGraph()

# Adicionar nos e arestas de todos os documentos
for i, (texto, grafo) in enumerate(zip(textos, grafos_extraidos)):
    # Adicionar entidades como nos
    for entidade in grafo.get('entities', []):
        if not G.has_node(entidade):
            # Embedar a entidade
            emb = model.encode(entidade, normalize_embeddings=True)
            G.add_node(entidade, embedding=emb, source_doc=i)
    
    # Adicionar relacoes como arestas
    for relacao in grafo.get('relations', []):
        if len(relacao) == 3:
            src, rel, tgt = relacao
            if src and tgt:
                G.add_edge(src, tgt, relation=rel)

# Fallback: se LLM nao disponivel, criar grafo manual
if G.number_of_nodes() == 0:
    print('LLM nao disponivel — usando grafo manual de demonstracao')
    entidades_manual = [
        ('HNSW', 'algoritmo'), ('Qdrant', 'banco_dados'), ('Embeddings', 'representacao'),
        ('BERT', 'modelo'), ('RAG', 'arquitetura'), ('Transformer', 'arquitetura'),
        ('LangChain', 'framework'), ('Busca_Semantica', 'tecnica'),
    ]
    relacoes_manual = [
        ('HNSW', 'indexa', 'Embeddings'), ('Qdrant', 'usa', 'HNSW'),
        ('BERT', 'gera', 'Embeddings'), ('BERT', 'e_um', 'Transformer'),
        ('RAG', 'usa', 'Busca_Semantica'), ('RAG', 'usa', 'Embeddings'),
        ('LangChain', 'implementa', 'RAG'), ('Qdrant', 'armazena', 'Embeddings'),
        ('Busca_Semantica', 'usa', 'Embeddings'),
    ]
    for ent, tipo in entidades_manual:
        emb = model.encode(ent, normalize_embeddings=True)
        G.add_node(ent, embedding=emb, tipo=tipo)
    for src, rel, tgt in relacoes_manual:
        G.add_edge(src, tgt, relation=rel)

print(f'Knowledge Graph: {G.number_of_nodes()} nos, {G.number_of_edges()} arestas')

In [ ]:
# Visualizar o grafo
fig, ax = plt.subplots(figsize=(14, 9))

pos = nx.spring_layout(G, seed=42, k=2)

nx.draw_networkx_nodes(G, pos, node_color='#3498db', node_size=1500, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, font_color='white', font_weight='bold', ax=ax)

nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True,
                       arrowsize=20, arrowstyle='->', connectionstyle='arc3,rad=0.1', ax=ax)

edge_labels = {(u, v): d.get('relation', '') for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax)

ax.set_title('Knowledge Graph — Entidades e Relacoes', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## 5.3 Graph-Augmented Retrieval

In [ ]:
def graph_retrieve(query: str, top_k: int = 5) -> dict:
    """
    Combina busca vetorial no grafo com expansao de vizinhos.
    1. Embeda a query
    2. Encontra entidades mais similares (cosine)
    3. Expande para vizinhos no grafo (hop-1)
    4. Retorna entidades + relacoes relevantes
    """
    q_emb = model.encode(query, normalize_embeddings=True)
    
    # Calcular similaridade com cada no
    node_scores = []
    for node, data in G.nodes(data=True):
        if 'embedding' in data:
            sim = float(q_emb @ data['embedding'])
            node_scores.append((node, sim))
    
    node_scores.sort(key=lambda x: x[1], reverse=True)
    top_nodes = node_scores[:top_k]
    
    # Expandir para vizinhos (hop-1)
    expanded_nodes = set(n for n, _ in top_nodes)
    for node, _ in top_nodes:
        expanded_nodes.update(G.predecessors(node))  # quem aponta para este no
        expanded_nodes.update(G.successors(node))    # para quem este no aponta
    
    # Extrair subgrafo relevante
    subgraph = G.subgraph(expanded_nodes)
    
    # Montar contexto do grafo
    relations_context = []
    for u, v, data in subgraph.edges(data=True):
        relations_context.append(f'{u} --[{data.get("relation", "relacionado")}]--> {v}')
    
    return {
        'top_entities': top_nodes,
        'expanded_entities': list(expanded_nodes),
        'relations': relations_context,
        'subgraph': subgraph,
    }

def graph_rag(query: str) -> str:
    """RAG com context enriquecido pelo knowledge graph."""
    graph_result = graph_retrieve(query)
    
    context = f"""Entidades mais relevantes para a query:
{chr(10).join(f'- {node} (score: {score:.3f})' for node, score in graph_result['top_entities'])}

Relacoes no conhecimento:
{chr(10).join(graph_result['relations'][:10])}
"""
    
    if LLM:
        prompt = f'Use o contexto do grafo de conhecimento para responder.\n\nContexto:\n{context}\n\nPergunta: {query}\n\nResposta:'
        response = ollama.chat(model=LLM, messages=[{'role': 'user', 'content': prompt}])
        return response['message']['content']
    else:
        return f'Contexto do grafo:\n{context}'

# Testar GraphRAG
queries_graph = [
    'Como o Qdrant usa HNSW para indexar embeddings?',
    'Qual e a relacao entre BERT e embeddings no RAG?',
]

for query in queries_graph:
    print(f'\nQuery: {query}')
    print('-'*50)
    
    result = graph_retrieve(query)
    print(f'Top entidades: {[(n, f"{s:.3f}") for n, s in result["top_entities"][:3]]}')
    print(f'Relacoes encontradas ({len(result["relations"])}): {result["relations"][:3]}')
    print()
    print(graph_rag(query)[:300])

## Resumo: Quando GraphRAG supera RAG tradicional?

| Tipo de Query | RAG Tradicional | GraphRAG |
|--------------|----------------|----------|
| Pergunta factual simples | ✅ Bom | ➡️ Similar |
| Relacoes entre entidades | ❌ Ruim | ✅ Excelente |
| Raciocinio multi-hop | ❌ Ruim | ✅ Excelente |
| Sintese de multiplos docs | ➡️ Moderado | ✅ Bom |
| Velocidade | ✅ Rapido | ❌ Mais lento |
| Custo de indexacao | ✅ Baixo | ❌ Alto (LLM extraction) |

## Proximo passo
- [05 PoCs](../05_pocs/README.md): Aplicar tudo num projeto completo